# 3D GCN: 3D features + Mean Aggregation (vs 3D GAT)

**Purpose**: Isolate the contribution of attention mechanism.
Same 3D features (23d edge, 11d node), same backbone, only difference: mean aggregation vs 12-head attention.

| Model | Features | Aggregation | Mean R2 |
|-------|----------|-------------|---------|
| GCN (2D) | 6d edge, 8d node | mean | 0.640 |
| GIN (2D) | 6d edge, 8d node | sum+MLP | 0.842 |
| 3D GCN | 23d edge, 11d node | mean | ? |
| 3D GAT | 23d edge, 11d node | 12-head attention | 0.903 |

30k subsample, 1 seed, ~30min.


In [1]:
import numpy as np, pandas as pd, time, gzip, os, json
import torch, torch.nn as nn, torch.optim as optim
from torch.amp import autocast, GradScaler
from collections import Counter
import warnings; warnings.filterwarnings("ignore")

os.environ.setdefault('OMP_NUM_THREADS', '16')
torch.set_num_threads(16)
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

from sklearn.metrics import r2_score
from tqdm import tqdm
from rdkit import Chem; from rdkit import RDLogger; RDLogger.logger().setLevel(RDLogger.ERROR)

df_y = pd.read_csv('tmQM_y.csv', sep=';')
df_y = df_y.rename(columns={'HL_Gap':'gap','HOMO_Energy':'homo','LUMO_Energy':'lumo','Dipole_M':'dipole','Metal_q':'metal_charge','Electronic_E':'E_elec','Dispersion_E':'E_disp','Polarizability':'polarizability'})
targets = ['gap','homo','lumo','dipole','metal_charge','E_elec','E_disp','polarizability']
print(f'Loaded {len(df_y)} complexes')


Device: cuda


Loaded 108541 complexes


In [2]:
# ===== Data loading (same as ablation_full.ipynb) =====
def atom_to_z(sym):
    tbl = {'H':1,'He':2,'Li':3,'Be':4,'B':5,'C':6,'N':7,'O':8,'F':9,'Ne':10,'Na':11,'Mg':12,'Al':13,'Si':14,'P':15,'S':16,'Cl':17,'Ar':18,'K':19,'Ca':20,'Sc':21,'Ti':22,'V':23,'Cr':24,'Mn':25,'Fe':26,'Co':27,'Ni':28,'Cu':29,'Zn':30,'Br':35,'Ru':44,'Rh':45,'Pd':46,'Ag':47,'Cd':48,'I':53,'Pt':78,'Au':79,'Hg':80,'Mo':42,'W':74,'Ir':77,'Os':76,'Re':75}
    return tbl.get(sym,0)
def is_TM(z):
    return z in [21,22,23,24,25,26,27,28,29,30,39,40,41,42,43,44,45,46,47,48,72,73,74,75,76,77,78,79,80]

def parse_one(xyz_p, bo_p, start_idx=0):
    mols, labels = [], []
    with gzip.open(bo_p, "rt") as f: bo_lines = [l.strip() for l in f.readlines()]
    with gzip.open(xyz_p, "rt") as f: xyz_blocks = f.read().strip().split(chr(10))
    bo_blocks, cur = [], []
    for line in bo_lines:
        if line.startswith("CSD_code"):
            if cur: bo_blocks.append(cur); cur = []
        elif line: cur.append(line)
    if cur: bo_blocks.append(cur)
    i, mol_idx = 0, start_idx - 1
    while i < len(xyz_blocks):
        line = xyz_blocks[i].strip(); i += 1
        if not line: continue
        parts = line.split()
        if len(parts)==1 and parts[0].isdigit():
            n_atoms = int(parts[0])
            if i >= len(xyz_blocks): break; i += 1
            atoms, coords = [], []
            for _ in range(n_atoms):
                if i >= len(xyz_blocks): break
                cline = xyz_blocks[i].strip(); i += 1; cparts = cline.split()
                if len(cparts)>=4:
                    try:
                        z = atom_to_z(cparts[0])
                        coords.append([float(x) for x in cparts[1:4]]); atoms.append(z)
                    except ValueError: continue
            mol_idx += 1
            if not any(is_TM(z) for z in atoms) or len(atoms)<=1: continue
            bo_mat = None
            if mol_idx < len(bo_blocks):
                bo_mat = np.zeros((n_atoms,n_atoms),dtype=np.float32)
                for bline in bo_blocks[mol_idx]:
                    bparts = bline.split()
                    try: src_idx=int(bparts[0])-1
                    except ValueError: continue
                    j = 2
                    while j+1<len(bparts):
                        try:
                            nbr=int(bparts[j])-1; bo_val=float(bparts[j+1])
                            if 0<=src_idx<n_atoms and 0<=nbr<n_atoms: bo_mat[src_idx,nbr]=bo_val
                            j+=3
                        except (ValueError,IndexError): j+=1
            mols.append({"atoms":np.array(atoms),"coords":np.array(coords,dtype=np.float32),"bo":bo_mat,"csv_idx":mol_idx})
            labels.append(df_y.iloc[mol_idx][targets].astype(float).values)
    return mols, labels

print('Parsing XYZ+BO...')
all_m, all_l = [], []
for part in [1,2,3]:
    m,l = parse_one(f"tmQM_X{part}.xyz.gz", f"tmQM_X{part}.BO.gz", start_idx=len(all_m))
    all_m.extend(m); all_l.extend(l)
print(f'Total: {len(all_m)} TM')

# SMILES graph building (same as ablation)
ELEMENT_DATA_SM = {1:(1,1),5:(2,13),6:(2,14),7:(2,15),8:(2,16),9:(2,17),14:(3,14),15:(3,15),16:(3,16),17:(3,17),20:(4,2),21:(4,3),22:(4,4),23:(4,5),24:(4,6),25:(4,7),26:(4,8),27:(4,9),28:(4,10),29:(4,11),30:(4,12),35:(4,17),42:(5,6),44:(5,8),45:(5,9),46:(5,10),47:(5,11),48:(5,12),53:(5,17),72:(6,4),73:(6,5),74:(6,6),75:(6,7),76:(6,8),77:(6,9),78:(6,10),79:(6,11),80:(6,12)}
BOND_TYPES_SM=[1,2,3,12]; B2I_SM={b:i for i,b in enumerate(BOND_TYPES_SM)}; N_BT_SM=4
rbf_c=np.linspace(1.0,5.0,16); rbf_sigma=0.2
smiles_by_idx=df_y["SMILES"].tolist()

def atom_features_sm(atom):
    z=atom.GetAtomicNum(); p,g=ELEMENT_DATA_SM.get(z,(0,0))
    return [z,atom.GetDegree(),atom.GetFormalCharge(),p,g,int(atom.GetIsAromatic()),atom.GetTotalNumHs(),int(z>=21)]

def fast_match(sm_atoms, xyz_atoms):
    N=len(sm_atoms)
    if N==len(xyz_atoms) and all(sm_atoms[i]==xyz_atoms[i] for i in range(N)): return list(range(N))
    sm_to_xyz=[-1]*N; used=set()
    for i,z_sm in enumerate(sm_atoms):
        for j,z_xyz in enumerate(xyz_atoms):
            if z_sm==z_xyz and j not in used: sm_to_xyz[i]=j; used.add(j); break
    return sm_to_xyz

# ===== FULL 95k — same data as 3D GAT ensemble CV =====
print(f'Building graphs for {len(all_m):,} complexes (FULL 95k)...')
graphs, y_vals, atoms_filt, failed = [], [], [], 0
for idx, m in enumerate(tqdm(all_m, desc="Building")):
    csv_i=m["csv_idx"]; smi=smiles_by_idx[csv_i] if csv_i<len(smiles_by_idx) else ""
    if not isinstance(smi,str) or smi=="": failed+=1; continue
    mol=Chem.MolFromSmiles(smi)
    if mol is None: failed+=1; continue
    N=mol.GetNumAtoms(); sm_atoms=[a.GetAtomicNum() for a in mol.GetAtoms()]
    sm_to_xyz=fast_match(sm_atoms, m["atoms"]); coords=m["coords"]; bo_mat=m.get("bo")
    xf=[atom_features_sm(a) for a in mol.GetAtoms()]
    if bo_mat is not None:
        for i in range(N):
            j=sm_to_xyz[i]
            if j>=0 and j<bo_mat.shape[0]:
                nb=int((bo_mat[j]>0.05).sum())-1; sb=float(bo_mat[j].sum())-bo_mat[j,j]; mb=float(bo_mat[j].max())
                xf[i].extend([max(0,nb),max(0,sb),max(0,mb)])
            else: xf[i].extend([0,0.0,0.0])
    else:
        for i in range(N): xf[i].extend([0,0.0,0.0])
    x=torch.tensor(xf,dtype=torch.float32); src,dst,battr=[],[],[]
    for bond in mol.GetBonds():
        i,j=bond.GetBeginAtomIdx(),bond.GetEndAtomIdx(); bt=int(bond.GetBondType())
        za,zb=sm_atoms[i],sm_atoms[j]; is_ml=float((za>=21)!=(zb>=21)); is_ring=float(bond.IsInRing())
        src.extend([i,j]); dst.extend([j,i])
        bf=[0.0]*N_BT_SM
        if bt in B2I_SM: bf[B2I_SM[bt]]=1.0
        bf.extend([is_ring,is_ml]); d,bo=2.0,0.0
        xi,xj=sm_to_xyz[i],sm_to_xyz[j]
        if xi>=0 and xj>=0 and xi<len(coords) and xj<len(coords): d=float(np.linalg.norm(coords[xi]-coords[xj]))
        if bo_mat is not None and xi>=0 and xj>=0 and xi<bo_mat.shape[0] and xj<bo_mat.shape[1]: bo=float(bo_mat[xi,xj])
        for mu in rbf_c: bf.append(np.exp(-(d-mu)**2/rbf_sigma))
        bf.append(bo); battr.extend([bf,bf])
    ei=torch.tensor([src,dst],dtype=torch.long) if src else torch.zeros(2,0,dtype=torch.long)
    ea=torch.tensor(battr,dtype=torch.float32) if battr else torch.zeros(0,23)
    graphs.append({'x':x,'edge_index':ei,'edge_attr':ea,'num_nodes':N})
    y_vals.append(all_l[idx]); atoms_filt.append(m["atoms"])

y_all=torch.tensor(y_vals,dtype=torch.float32)
mask=~(torch.isnan(y_all).any(dim=1)|torch.isinf(y_all).any(dim=1))
graphs=[g for i,g in enumerate(graphs) if mask[i]]; y_all=y_all[mask]; atoms_filt=[a for i,a in enumerate(atoms_filt) if mask[i]]
print(f'Built {len(graphs):,} graphs ({failed} failed)')

y_mean=y_all.mean(dim=0); y_std=y_all.std(dim=0).clamp(min=1e-8); y_norm=(y_all-y_mean)/y_std
# ===== SAME split as 3D GAT (ensemble_split.npz) =====
if os.path.exists('ensemble_split.npz'):
    split = np.load('ensemble_split.npz')
    tr_idx = split['fold_0_train']; va_idx = split['fold_0_val']; te_idx = split['te_idx']
    print(f'Using ensemble_split.npz (same test set as 3D GAT)')
else:
    idx=np.random.RandomState(42).permutation(len(graphs))
    n_tr=int(len(graphs)*0.7); n_va=int(len(graphs)*0.15)
    tr_idx=idx[:n_tr]; va_idx=idx[n_tr:n_tr+n_va]; te_idx=idx[n_tr+n_va:]
print(f'Train:{len(tr_idx):,} Val:{len(va_idx):,} Test:{len(te_idx):,}')


Parsing XYZ+BO...


Total: 96984 TM
Building graphs for 96,984 complexes (FULL 95k)...


Building:   0%|          | 0/96984 [00:00<?, ?it/s]

Building:   0%|          | 40/96984 [00:00<04:09, 388.88it/s]

Building:   0%|          | 79/96984 [00:00<04:21, 370.35it/s]

...

Building: 100%|█████████▉| 96918/96984 [04:53<00:00, 296.18it/s]

Building: 100%|█████████▉| 96949/96984 [04:53<00:00, 283.84it/s]

Building: 100%|█████████▉| 96979/96984 [04:53<00:00, 280.91it/s]

Building: 100%|██████████| 96984/96984 [04:53<00:00, 330.41it/s]

Built 90,317 graphs (6667 failed)
Using ensemble_split.npz (same test set as 3D GAT)
Train:51,180 Val:25,590 Test:13,547


In [3]:
# ===== Model: 3D GCN = mean aggregation + 3D features =====
HIDDEN = 384; N_LAYERS = 5; BATCH = 128; EPOCHS = 150; EARLY_STOP = 40

class GCNConv(nn.Module):
    """Mean aggregation (no attention, no edge features in message)"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, x, ei, ea=None):
        src, dst = ei; N = x.size(0)
        msg = x[src]
        aggr = torch.zeros(N, x.size(1), device=x.device, dtype=x.dtype)
        aggr.scatter_add_(0, dst.unsqueeze(-1).expand_as(msg), msg)
        deg = torch.zeros(N, device=x.device, dtype=x.dtype)
        deg.scatter_add_(0, dst, torch.ones_like(src, dtype=x.dtype))
        deg = deg.clamp(min=1)
        return self.lin(aggr / deg.unsqueeze(-1))

def collate_graphs(graph_list):
    x_list, ei_list, ea_list, batch_list = [], [], [], []
    offset = 0
    for i, g in enumerate(graph_list):
        n = g["x"].size(0)
        x_list.append(g["x"]); ei_list.append(g["edge_index"] + offset)
        ea_list.append(g["edge_attr"])
        batch_list.append(torch.full((n,), i, dtype=torch.long, device=g["x"].device))
        offset += n
    return {'x': torch.cat(x_list, dim=0), 'edge_index': torch.cat(ei_list, dim=1), 'edge_attr': torch.cat(ea_list, dim=0)}, torch.cat(batch_list, dim=0)

def scatter_mean_per_graph(h, batch, n):
    out = torch.zeros(n, h.size(1), device=h.device, dtype=h.dtype)
    out.scatter_add_(0, batch.unsqueeze(-1).expand_as(h), h)
    cnt = torch.zeros(n, device=h.device, dtype=h.dtype)
    cnt.scatter_add_(0, batch, torch.ones_like(batch, dtype=h.dtype))
    return out / cnt.clamp(min=1).unsqueeze(-1)

def scatter_max_per_graph(h, batch, n):
    out = torch.full((n, h.size(1)), float('-inf'), device=h.device, dtype=h.dtype)
    out.scatter_reduce_(0, batch.unsqueeze(-1).expand_as(h), h, reduce='amax', include_self=False)
    return out

class ThreeD_GCN(nn.Module):
    def __init__(self, node_dim=11, hidden=HIDDEN, n_layers=N_LAYERS, output_dim=8, dropout=0.10):
        super().__init__()
        self.embed = nn.Linear(node_dim, hidden)
        self.convs = nn.ModuleList([GCNConv(hidden, hidden) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.readout = nn.Sequential(
            nn.Linear(hidden*3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden//2, output_dim))
    def forward(self, g, batch=None):
        x, ei = g['x'], g['edge_index']
        h = self.embed(x)
        for conv, norm in zip(self.convs, self.norms): h = norm(h + conv(h, ei)); h = self.drop(h)
        if batch is None:
            mm = (x[:,7] > 0.5); mf = h[mm].mean(dim=0) if mm.any() else torch.zeros(h.size(1), device=h.device, dtype=h.dtype)
            return self.readout(torch.cat([h.mean(0), h.max(0).values, mf])).unsqueeze(0)
        n = int(batch.max().item()) + 1
        mean_h = scatter_mean_per_graph(h, batch, n); max_h = scatter_max_per_graph(h, batch, n)
        mm = (x[:,7] > 0.5); mf = scatter_mean_per_graph(h[mm], batch[mm], n) if mm.any() else torch.zeros(n, h.size(1), device=h.device, dtype=h.dtype)
        return self.readout(torch.cat([mean_h, max_h, mf], dim=1))

m = ThreeD_GCN().to(device)
print(f'3D GCN params: {sum(p.numel() for p in m.parameters()):,}')


3D GCN params: 1,263,944


In [4]:
# ===== Training =====
torch.manual_seed(42); np.random.seed(42)
model = ThreeD_GCN().to(device)
opt = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2, eta_min=1e-6)
crit = nn.MSELoss()
scaler = GradScaler('cuda')
yn = y_norm.to(device)

# GPU preload (30k fits easily)
print('Pre-loading graphs to GPU...')
graphs_gpu = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in g.items()} for g in tqdm(graphs, desc='GPU upload')]

best_va = float("inf"); best_state = None; patience = 0
print(f'Training {EPOCHS} epochs...')
for ep in range(EPOCHS):
    model.train(); perm = torch.randperm(len(tr_idx))
    for s in range(0, len(tr_idx), BATCH):
        bidx = tr_idx[perm[s:s+BATCH]].tolist()
        bg = [graphs_gpu[i] for i in bidx]; by = yn[bidx]
        batched, batch_idx = collate_graphs(bg)
        opt.zero_grad()
        with autocast('cuda'): loss = crit(model(batched, batch_idx), by)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
    model.eval()
    with torch.no_grad():
        vp = []
        for s in range(0, len(va_idx), BATCH):
            bidx = va_idx[s:s+BATCH].tolist()
            bg = [graphs_gpu[i] for i in bidx]
            batched, batch_idx = collate_graphs(bg)
            with autocast('cuda'): vp.append(model(batched, batch_idx).float())
        va_loss = nn.MSELoss()(torch.cat(vp), yn[va_idx])
    sched.step(ep + va_loss.item()/10)
    if va_loss < best_va:
        best_va = va_loss.item(); best_state = {k:v.clone() for k,v in model.state_dict().items()}; patience = 0
    else: patience += 1
    if patience >= EARLY_STOP:
        print(f'Early stop @ ep {ep+1}'); break
    if (ep+1) % 30 == 0:
        print(f'Ep {ep+1}: va={va_loss.item():.4f} pat={patience}')

model.load_state_dict(best_state); model.eval()



Pre-loading graphs to GPU...


GPU upload:   0%|          | 0/90317 [00:00<?, ?it/s]

GPU upload:   2%|▏         | 1569/90317 [00:00<00:05, 15682.39it/s]

GPU upload:   3%|▎         | 3138/90317 [00:00<00:05, 15509.25it/s]

GPU upload:   5%|▌         | 4690/90317 [00:00<00:05, 15109.73it/s]

...

Ep 60: va=0.7569 pat=1


Ep 90: va=0.5563 pat=6


Ep 120: va=0.4545 pat=10


Ep 150: va=0.4534 pat=12


ThreeD_GCN(
  (embed): Linear(in_features=11, out_features=384, bias=True)
  (convs): ModuleList(
    (0-4): 5 x GCNConv(
      (lin): Linear(in_features=384, out_features=384, bias=False)
    )
  )
  (norms): ModuleList(
    (0-4): 5 x LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (drop): Dropout(p=0.1, inplace=False)
  (readout): Sequential(
    (0): Linear(in_features=1152, out_features=384, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=384, out_features=192, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=192, out_features=8, bias=True)
  )
)

In [5]:
# ===== Test evaluation =====
with torch.no_grad():
    tp = []
    for s in range(0, len(te_idx), BATCH):
        bidx = te_idx[s:s+BATCH].tolist()
        bg = [graphs_gpu[i] for i in bidx]
        batched, batch_idx = collate_graphs(bg)
        with autocast('cuda'): tp.append(model(batched, batch_idx).float().cpu())
    tp = torch.cat(tp)
tp_orig = tp * y_std + y_mean; tt = y_all[te_idx]

print(f'{"Property":16s} {"MAE":>10s} {"R2":>8s}')
print('-'*38)
results = []
for i, prop in enumerate(targets):
    yt, yp = tt[:,i].numpy(), tp_orig[:,i].numpy()
    mae = float(np.mean(np.abs(yt-yp))); r2 = float(r2_score(yt, yp))
    print(f'{prop:16s} {mae:10.4f} {r2:8.4f}')
    results.append({'Property':prop,'MAE':mae,'R2':r2})
mean_r2 = np.mean([r['R2'] for r in results])
print(f'\n3D GCN Mean R2: {mean_r2:.4f}')

with open('gcn_3d_results.json', 'w') as f:
    json.dump({'results': results, 'mean_r2': mean_r2}, f, indent=2)
print('Saved gcn_3d_results.json')


Property                MAE       R2
--------------------------------------
gap                  0.0139   0.6441
homo                 0.0161   0.7798
lumo                 0.0166   0.7726
dipole               2.1166   0.3922
metal_charge         0.1911   0.8751
E_elec             580.5380   0.7120
E_disp               0.0469   0.1950
polarizability     104.3732   0.1921

3D GCN Mean R2: 0.5704
Saved gcn_3d_results.json


## Interpretation

| Model | Dim | Aggregation | R2 |
|-------|-----|-------------|-----|
| GCN (2D) | 8d/6d | mean | 0.640 |
| GIN (2D) | 8d/6d | sum+MLP | 0.842 |
| **3D GCN** | 11d/23d | mean | ? |
| **3D GAT** | 11d/23d | 12-head attention | 0.903 |

Gap between 3D GCN and 3D GAT = contribution of attention mechanism given identical 3D features.
